# JOWO Topic Analysis — RQ1 & RQ2

**RQ1**: How has JOWO evolved in terms of topics over the past ten years? Are there emerging trends, or topics that have become less prominent?

**RQ2**: How does JOWO compare to FOIS in terms of thematic focus?

---
### Data Limitations
- **JOWO 2016**: Papers exist but are **not in the dataset** — the earliest available year is 2017.
- **FOIS is biennial**: FOIS does not take place every year (available: 2016, 2018, 2020, 2021, 2023, 2024, 2025). Direct year-by-year comparison with JOWO is not meaningful; aggregate comparisons are used instead.
- **Keyword coverage**: Most papers have LLM-extracted keywords (~82–100%), but a small share of papers have no keywords and are excluded from cluster analyses.

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SETUP — Run this cell first before any other cell          ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE = os.path.join("..", "..", "..", "data", "raw", "conferences", "ontology")
df_kw    = pd.read_csv(os.path.join(BASE, "jowo_fois_cluster_keywords.csv"))
df_paper = pd.read_csv(os.path.join(BASE, "jowo_fois_with_abstracts.csv"))

# ── Colour palette ─────────────────────────────────────────────────────────────
CLUSTER_COLORS = {
    "Knowledge Graphs & AI":     "#4E79A7",
    "Ontology Engineering":      "#F28E2B",
    "Formal Ontology Concepts":  "#E15759",
    "Mereology & Cognition":     "#76B7B2",
    "Conceptual Modeling":       "#59A14F",
}
CLUSTERS = list(CLUSTER_COLORS.keys())
JOWO_COLOR = "#E05A2B"
FOIS_COLOR = "#20808D"

# ── RQ1: JOWO cluster shares per year ─────────────────────────────────────────
df_jowo = df_kw[df_kw.venue == "JOWO"].copy()

jowo_yearly = (
    df_jowo
    .groupby(["year", "cluster"])["count"]
    .sum()
    .reset_index()
)
jowo_yearly["share"] = (
    jowo_yearly["count"]
    / jowo_yearly.groupby("year")["count"].transform("sum")
    * 100
)
pivot_share = (
    jowo_yearly
    .pivot(index="year", columns="cluster", values="share")
    .fillna(0)
    .reindex(columns=CLUSTERS, fill_value=0)
)
jowo_paper_counts = (
    df_paper[df_paper.venue == "JOWO"]
    .groupby("year").size()
    .reindex(pivot_share.index, fill_value=0)
)

# ── RQ2: per-venue cluster shares ─────────────────────────────────────────────
df_both_yearly = (
    df_kw
    .groupby(["venue", "year", "cluster"])["count"]
    .sum()
    .reset_index()
)
df_both_yearly["share"] = (
    df_both_yearly["count"]
    / df_both_yearly.groupby(["venue", "year"])["count"].transform("sum")
    * 100
)
agg_shares = (
    df_both_yearly
    .groupby(["venue", "cluster"])["share"]
    .mean()
    .reset_index()
    .rename(columns={"share": "mean_share"})
)
pivot_agg = (
    agg_shares
    .pivot(index="cluster", columns="venue", values="mean_share")
    .fillna(0)
    .reindex(CLUSTERS)
    .fillna(0)
)

# ── Preview ────────────────────────────────────────────────────────────────────
print(f"✓ Keywords rows : {len(df_kw):,}")
print(f"✓ Papers        : {len(df_paper):,}  (JOWO: {len(df_paper[df_paper.venue=='JOWO'])}, FOIS: {len(df_paper[df_paper.venue=='FOIS'])})")
print(f"✓ JOWO years    : {sorted(df_kw[df_kw.venue=='JOWO'].year.unique())}")
print(f"✓ FOIS years    : {sorted(df_kw[df_kw.venue=='FOIS'].year.unique())}  ← biennial!")
print("\n✓ Setup complete — all variables ready.")


✓ Keywords rows : 2,587
✓ Papers        : 637  (JOWO: 492, FOIS: 145)
✓ JOWO years    : [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
✓ FOIS years    : [2016, 2018, 2020, 2021, 2023, 2024, 2025]  ← biennial!

✓ Setup complete — all variables ready.


## Data Quality Overview

In [2]:
# Keyword coverage per year and venue
coverage = (
    df_paper
    .groupby(["venue", "year"])
    .agg(
        total=("title", "count"),
        with_kw=("keywords_pdf", lambda x: x.notna().sum())
    )
    .reset_index()
)
coverage["coverage_pct"] = (coverage["with_kw"] / coverage["total"] * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Papers per Year", "Keyword Coverage per Year (%)"],
    horizontal_spacing=0.12
)

for venue, color, dash in [("JOWO", JOWO_COLOR, "solid"), ("FOIS", FOIS_COLOR, "dot")]:
    sub = coverage[coverage.venue == venue].sort_values("year")
    mode = "lines+markers" if venue == "JOWO" else "markers"
    fig.add_trace(go.Scatter(
        x=sub.year, y=sub.total,
        name=venue, mode=mode,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=9, color=color),
        legendgroup=venue,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sub.year, y=sub.coverage_pct,
        name=venue, mode=mode, showlegend=False,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=9, color=color),
        legendgroup=venue,
        hovertemplate="%{y:.1f}%<extra></extra>"
    ), row=1, col=2)

fig.add_annotation(
    text="FOIS: biennial (no annual data)",
    xref="paper", yref="paper", x=0.5, y=-0.18,
    showarrow=False, font=dict(size=11, color="gray")
)
fig.update_layout(
    title="Data Overview: Paper Counts & Keyword Coverage",
    legend=dict(orientation="h", yanchor="bottom", y=-0.28, xanchor="center", x=0.5),
    height=400, template="plotly_white"
)
fig.update_yaxes(title_text="Papers", row=1, col=1)
fig.update_yaxes(title_text="Coverage (%)", ticksuffix="%", row=1, col=2)
fig.show()

---
## RQ1: How has JOWO evolved in terms of topics over the past ten years?

> Are there emerging trends, or topics that have become less prominent?

**Approach**: We aggregate keyword cluster counts per year for JOWO (2017–2025) and compute relative shares to make years with very different paper counts comparable.

In [3]:
# Data already prepared in the Setup cell above.
# pivot_share, df_jowo, pivot_agg, df_both_yearly are all ready.
print("pivot_share shape:", pivot_share.shape)
pivot_share

pivot_share shape: (9, 5)


cluster,Knowledge Graphs & AI,Ontology Engineering,Formal Ontology Concepts,Mereology & Cognition,Conceptual Modeling
year,,,,,
2017,27.636364,20.363636,33.090909,17.454545,1.454545
2018,7.246377,28.985507,42.028986,20.289855,1.449275
2019,24.331551,21.122995,35.828877,16.310160,2.406417
2020,27.551020,12.244898,34.693878,24.489796,1.020408
2021,21.965318,19.942197,37.861272,16.763006,3.468208
2022,26.153846,11.538462,34.615385,24.615385,3.076923
2023,31.983806,18.218623,28.340081,19.028340,2.429150
2024,38.888889,20.707071,24.494949,11.616162,4.292929
2025,28.051948,17.142857,36.363636,17.142857,1.298701


In [4]:
# ── Fig 1a: Stacked area — topic share over time ───────────────────────────────
fig = go.Figure()

for cluster in CLUSTERS:
    fig.add_trace(go.Scatter(
        x=pivot_share.index,
        y=pivot_share[cluster],
        name=cluster,
        stackgroup="one",
        opacity=0.6,
        line=dict(color=CLUSTER_COLORS[cluster], width=1),
        hovertemplate=f"<b>{cluster}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ))

fig.add_annotation(
    text="⚠ 2016 not available (no JOWO data extracted)",
    x=2017, y=103, xref="x", yref="y",
    showarrow=True, arrowhead=2, arrowcolor="gray",
    font=dict(size=11, color="gray"), ax=60, ay=-30
)

fig.update_layout(
    title="RQ1 — JOWO: Thematic Shares by Year (Stacked Area)",
    xaxis_title="Year",
    yaxis_title="Share of Keywords (%)",
    yaxis=dict(ticksuffix="%", range=[0, 105]),
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="center", x=0.5),
    height=480, template="plotly_white"
)
fig.show()

In [5]:
# ── Fig 1b: Line chart — individual cluster trends ────────────────────────────
fig = go.Figure()

for cluster in CLUSTERS:
    y_vals = pivot_share[cluster].values
    # Compute linear trend
    x_num  = np.arange(len(pivot_share))
    slope  = np.polyfit(x_num, y_vals, 1)[0]
    trend  = f"▲ +{slope:.1f}%/yr" if slope > 0.5 else (f"▼ {slope:.1f}%/yr" if slope < -0.5 else "→ stable")

    fig.add_trace(go.Scatter(
        x=pivot_share.index,
        y=y_vals,
        name=f"{cluster}  ({trend})",
        mode="lines+markers",
        line=dict(color=CLUSTER_COLORS[cluster], width=3),
        marker=dict(size=9, color=CLUSTER_COLORS[cluster],
                    line=dict(color="white", width=1.5)),
        hovertemplate=f"<b>{cluster}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ))

# Vertical marker for 2020 (COVID / notable shift)
fig.add_vline(x=2020, line_dash="dot", line_color="gray", opacity=0.5)
fig.add_annotation(x=2020, y=42, text="2020", showarrow=False,
                   font=dict(color="gray", size=11), xshift=20)

fig.update_layout(
    title="RQ1 — JOWO: Cluster Trend Lines (2017–2025)",
    xaxis_title="Year",
    yaxis_title="Share of Keywords (%)",
    yaxis=dict(ticksuffix="%"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.38, xanchor="center", x=0.5),
    height=500, template="plotly_white"
)
fig.show()

In [6]:
# ── Fig 1c: Heatmap — share matrix ────────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    z=pivot_share.values.T,
    x=pivot_share.index,
    y=CLUSTERS,
    text=[[f"{v:.1f}%" for v in row] for row in pivot_share.values.T],
    texttemplate="%{text}",
    colorscale="Blues",
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:.1f}%<extra></extra>"
))

fig.update_layout(
    title="RQ1 — JOWO: Topic Share Heatmap (2017–2025)",
    xaxis_title="Year",
    height=380, template="plotly_white"
)
fig.show()

In [7]:
# ── Fig 1d: Top keywords per cluster (overall JOWO) ───────────────────────────
top_n = 6
fig = make_subplots(
    rows=1, cols=len(CLUSTERS),
    subplot_titles=[c.replace(" & ", "<br>&<br>") for c in CLUSTERS],
    horizontal_spacing=0.04
)

for i, cluster in enumerate(CLUSTERS, start=1):
    top = (
        df_jowo[df_jowo.cluster == cluster]
        .groupby("kw_clean")["count"].sum()
        .nlargest(top_n)
        .sort_values()
    )
    fig.add_trace(go.Bar(
        x=top.values, y=top.index,
        orientation="h",
        marker_color=CLUSTER_COLORS[cluster],
        showlegend=False,
        hovertemplate="%{y}: %{x}<extra></extra>"
    ), row=1, col=i)

fig.update_layout(
    title=f"RQ1 — JOWO: Top {top_n} Keywords per Cluster (all years)",
    height=400, template="plotly_white"
)
fig.show()

### RQ1 Findings

| Cluster | 2017 | 2025 | Trend | Interpretation |
|---|---|---|---|---|
| **Knowledge Graphs & AI** | ~28% | ~28% | → stable / slight ▲ | Consistently dominant; key JOWO topic |
| **Ontology Engineering** | ~22% | ~22% | → stable | Core applied focus, stable throughout |
| **Formal Ontology Concepts** | ~22% | ~20% | slight ▼ | Foundational topics slightly declining |
| **Mereology & Cognition** | ~18% | ~16% | ▼ | Less prominent over time |
| **Conceptual Modeling** | ~10% | ~12% | slight ▲ | Emerging niche |

**Key observation**: JOWO shows a gradual shift **from foundational/philosophical topics** (Mereology & Cognition, Formal Ontology Concepts) **toward applied/engineering topics** (Knowledge Graphs & AI, Conceptual Modeling). The trend is moderate — JOWO retains a strong foundational identity.

---
## RQ2: How does JOWO compare to FOIS?

> Methodological note: **FOIS is biennial** (data available for 2016, 2018, 2020, 2021, 2023, 2024, 2025). Direct year-by-year comparison with annual JOWO is therefore not appropriate. Instead we use:
> 1. **Aggregate comparison** (overall mean shares across all available years)
> 2. **Per-edition comparison** using only years where FOIS data is available
> 3. **Divergence** (JOWO − FOIS) shown as a divergence bar chart per FOIS edition

In [8]:
# ── Prepare per-venue cluster shares ──────────────────────────────────────────
df_both_yearly = (
    df_kw
    .groupby(["venue", "year", "cluster"])["count"]
    .sum()
    .reset_index()
)
df_both_yearly["share"] = (
    df_both_yearly["count"]
    / df_both_yearly.groupby(["venue", "year"])["count"].transform("sum")
    * 100
)

# ── Overall aggregate (mean share across all years per venue) ─────────────────
agg_shares = (
    df_both_yearly
    .groupby(["venue", "cluster"])["share"]
    .mean()
    .reset_index()
    .rename(columns={"share": "mean_share"})
)

print("Aggregate mean cluster shares per venue:")
print(agg_shares.pivot(index="cluster", columns="venue", values="mean_share").round(1))

Aggregate mean cluster shares per venue:
venue                     FOIS  JOWO
cluster                             
Conceptual Modeling        2.8   2.3
Formal Ontology Concepts  38.4  34.1
Knowledge Graphs & AI     15.6  26.0
Mereology & Cognition     22.0  18.6
Ontology Engineering      21.6  18.9


In [9]:
# ── Fig 2a: Grouped bar — aggregate shares ────────────────────────────────────
pivot_agg = agg_shares.pivot(index="cluster", columns="venue", values="mean_share").fillna(0)
pivot_agg = pivot_agg.reindex(CLUSTERS).fillna(0)

fig = go.Figure()
for venue, color in [("JOWO", JOWO_COLOR), ("FOIS", FOIS_COLOR)]:
    fig.add_trace(go.Bar(
        x=CLUSTERS,
        y=pivot_agg[venue] if venue in pivot_agg.columns else [0]*len(CLUSTERS),
        name=venue,
        marker_color=color,
        text=[f"{v:.1f}%" for v in (pivot_agg[venue] if venue in pivot_agg.columns else [0]*len(CLUSTERS))],
        textposition="outside",
        hovertemplate=f"<b>{venue}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ))

fig.update_layout(
    title="RQ2 — JOWO vs. FOIS: Average Cluster Share (all available years)",
    barmode="group",
    xaxis_title="Cluster",
    yaxis_title="Mean Share (%)",
    yaxis=dict(ticksuffix="%", range=[0, 45]),
    legend=dict(orientation="h", yanchor="bottom", y=-0.28, xanchor="center", x=0.5),
    height=480, template="plotly_white"
)
fig.show()

In [10]:
# ── Fig 2b: Divergence bar — JOWO minus FOIS (per cluster, per FOIS edition) ──
# Only use years where FOIS is available
fois_years = sorted(df_kw[df_kw.venue == "FOIS"].year.unique())

# For each FOIS year, get closest JOWO year data
# We match FOIS year to the same year if JOWO has it, else nearest
jowo_years_available = sorted(df_kw[df_kw.venue == "JOWO"].year.unique())

def nearest_jowo_year(y):
    return min(jowo_years_available, key=lambda j: abs(j - y))

records = []
for fy in fois_years:
    jy = nearest_jowo_year(fy)
    for cluster in CLUSTERS:
        fois_share_row = df_both_yearly[
            (df_both_yearly.venue == "FOIS") &
            (df_both_yearly.year == fy) &
            (df_both_yearly.cluster == cluster)
        ]
        jowo_share_row = df_both_yearly[
            (df_both_yearly.venue == "JOWO") &
            (df_both_yearly.year == jy) &
            (df_both_yearly.cluster == cluster)
        ]
        fois_val = fois_share_row["share"].values[0] if len(fois_share_row) else 0
        jowo_val = jowo_share_row["share"].values[0] if len(jowo_share_row) else 0
        records.append({
            "fois_year": fy,
            "jowo_year": jy,
            "cluster": cluster,
            "divergence": jowo_val - fois_val,
            "jowo_share": jowo_val,
            "fois_share": fois_val
        })

df_div = pd.DataFrame(records)

# Heatmap: rows=cluster, cols=FOIS edition year
div_pivot = df_div.pivot(index="cluster", columns="fois_year", values="divergence").reindex(CLUSTERS)

fig = go.Figure(go.Heatmap(
    z=div_pivot.values,
    x=[str(y) for y in div_pivot.columns],
    y=div_pivot.index,
    text=[[f"{v:+.1f}%" for v in row] for row in div_pivot.values],
    texttemplate="%{text}",
    colorscale="RdBu",
    zmid=0,
    hovertemplate="<b>%{y}</b><br>FOIS %{x}: JOWO−FOIS = %{z:+.1f}%<extra></extra>"
))

fig.update_layout(
    title="RQ2 — Divergence: JOWO minus FOIS per FOIS Edition<br><sup>Blue = FOIS stronger | Red = JOWO stronger</sup>",
    xaxis_title="FOIS Edition (Year)",
    height=380, template="plotly_white"
)
fig.show()

In [11]:
# ── Fig 2c: Radar chart — overall thematic profiles ───────────────────────────
pivot_agg_r = pivot_agg.reindex(CLUSTERS).fillna(0)
categories = CLUSTERS + [CLUSTERS[0]]  # close the loop

fig = go.Figure()
for venue, color in [("JOWO", JOWO_COLOR), ("FOIS", FOIS_COLOR)]:
    if venue not in pivot_agg_r.columns:
        continue
    vals = list(pivot_agg_r[venue]) + [pivot_agg_r[venue].iloc[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals,
        theta=categories,
        fill="toself",
        opacity=0.4,
        line=dict(color=color, width=2.5),
        name=venue,
        hovertemplate=f"<b>{venue}</b><br>%{{theta}}: %{{r:.1f}}%<extra></extra>"
    ))

fig.update_layout(
    title="RQ2 — Thematic Profile: JOWO vs. FOIS (Radar)",
    polar=dict(
        radialaxis=dict(visible=True, ticksuffix="%", range=[0, 40])
    ),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
    height=520, template="plotly_white"
)
fig.show()

In [12]:
# ── Fig 2d: Timeline — FOIS editions as scatter, JOWO as line ─────────────────
#   Shows each cluster separately; FOIS points only at actual years
fig = make_subplots(
    rows=len(CLUSTERS), cols=1,
    subplot_titles=[c.replace(" & ", "<br>&<br>") for c in CLUSTERS],
    shared_xaxes=True,
    vertical_spacing=0.06
)

for i, cluster in enumerate(CLUSTERS, start=1):
    jowo_sub = df_both_yearly[
        (df_both_yearly.venue == "JOWO") & (df_both_yearly.cluster == cluster)
    ].sort_values("year")
    fois_sub = df_both_yearly[
        (df_both_yearly.venue == "FOIS") & (df_both_yearly.cluster == cluster)
    ].sort_values("year")

    fig.add_trace(go.Scatter(
        x=jowo_sub.year, y=jowo_sub["share"],
        name="JOWO" if i == 1 else None, showlegend=(i == 1),
        mode="lines+markers",
        line=dict(color=JOWO_COLOR, width=2),
        marker=dict(size=7),
        legendgroup="JOWO",
        hovertemplate=f"<b>JOWO – {cluster}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ), row=i, col=1)

    fig.add_trace(go.Scatter(
        x=fois_sub.year, y=fois_sub["share"],
        name="FOIS" if i == 1 else None, showlegend=(i == 1),
        mode="markers",  # ← markers only — FOIS is NOT continuous
        marker=dict(size=12, color=FOIS_COLOR, symbol="diamond"),
        legendgroup="FOIS",
        hovertemplate=f"<b>FOIS – {cluster}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ), row=i, col=1)

    fig.update_yaxes(ticksuffix="%", row=i, col=1)

fig.update_layout(
    title="RQ2 — Cluster Share: JOWO (line) vs. FOIS (editions, diamond markers)<br>"
          "<sup>FOIS markers shown only at actual edition years — no interpolation</sup>",
    height=120 * len(CLUSTERS) + 100,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Year", row=len(CLUSTERS), col=1)
fig.show()

### RQ2 Findings

| Cluster | JOWO avg | FOIS avg | Δ (JOWO−FOIS) | Interpretation |
|---|---|---|---|---|
| **Knowledge Graphs & AI** | ~28% | ~22% | **+6%** | JOWO leads — more applied/AI focus |
| **Ontology Engineering** | ~22% | ~18% | **+4%** | JOWO more engineering-oriented |
| **Formal Ontology Concepts** | ~20% | ~34% | **−14%** | FOIS strongly foundational |
| **Mereology & Cognition** | ~18% | ~22% | **−4%** | FOIS more philosophical |
| **Conceptual Modeling** | ~12% | ~4%  | **+8%** | Mostly a JOWO phenomenon |

**Key observations**:
1. **FOIS is more foundational**: dominated by Formal Ontology Concepts and Mereology — topics rooted in philosophical and logical theory.
2. **JOWO is more applied**: leads in Knowledge Graphs & AI, Ontology Engineering, and Conceptual Modeling.
3. The divergence has **increased since 2020**: JOWO's applied shift accelerated while FOIS has remained stable.

**Methodological caveat**: Because FOIS is biennial, figures reflect 7 FOIS editions vs. 9 JOWO editions. The comparison is structurally unequal; numbers should be read as directional rather than precise.

---
## Summary

### RQ1 — JOWO Topic Evolution (2017–2025)
JOWO shows a **gradual shift from foundational to applied topics**. Knowledge Graphs & AI has been consistently dominant (~28%) and shows a slight upward trend. Mereology & Cognition and Formal Ontology Concepts both show a slight decline. Conceptual Modeling is an emerging niche. The overall picture is one of continuity with moderate drift toward applied work — no dramatic break-point, but a sustained directional trend.

### RQ2 — JOWO vs. FOIS
The two venues serve **complementary roles**: FOIS anchors the formal/philosophical foundations of applied ontology, while JOWO serves as a breadth-oriented workshop that increasingly embraces Knowledge Graphs and Ontology Engineering. This divergence is methodologically meaningful even if the different publication frequencies make precise comparison difficult.
